# 5-Fold Cross-Validation и OOF Blending

**Методология:**
1. Обучающая выборка (`train`) разбивается на одни и те же 5 стратифицированных фолдов (`StratifiedKFold`).
2. Каждая модель делает Out-Of-Fold (OOF) предсказания — каждый объект предсказывается моделью, которая не видела его при обучении.
3. Считаем `mean` и `std` ROC-AUC по 5 фолдам для каждой отдельной модели.
4. Ансамбли (бленды) оцениваем **только** по OOF предсказаниям фолд-за-фолдом.
5. Валидационная выборка больше не используется для выбора гиперпараметров и бленда во избежание data leakage / оверфита на holdout.


### 1. Данные и генерация признаков (Raw + Pair)


In [1]:
import os
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")

SEED = 20260916

DATA_DIR = next(p for p in [".", "..", "../.."] if os.path.exists(f"{p}/train.csv"))

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_submission = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

feature_columns = [
    c for c in train.columns
    if c not in ["row_id", "target"]
]

X_raw = train[feature_columns].to_numpy()
X_test_raw = test[feature_columns].to_numpy()

y = train["target"].to_numpy()


def make_pair_features(X):
    A = X[:, :256]
    B = X[:, 256:]

    abs_diff = np.abs(A - B)

    return np.hstack([
        X,
        abs_diff,
    ])


X_pair = make_pair_features(X_raw)
X_test_pair = make_pair_features(X_test_raw)

feature_sets = {
    "raw": X_raw,
    "pair": X_pair,
}

test_feature_sets = {
    "raw": X_test_raw,
    "pair": X_test_pair,
}

print("raw:", X_raw.shape)
print("pair:", X_pair.shape)


raw: (2520, 512)
pair: (2520, 768)


### 2. Фиксированные 5 folds для всех моделей


In [2]:
cv = list(
    StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=SEED,
    ).split(X_raw, y)
)

fold_id = np.zeros(len(y), dtype=int)

for fold, (_, val_idx) in enumerate(cv):
    fold_id[val_idx] = fold


### 3. Спецификации всех исследуемых моделей


In [3]:
model_specs = [
    {
        "name": "Decision Tree",
        "features": "raw",
        "factory": lambda: DecisionTreeClassifier(
            criterion="entropy",
            max_depth=10,
            max_features=0.3,
            min_samples_leaf=29,
            min_samples_split=2,
            splitter="best",
            ccp_alpha=0.005,
            random_state=SEED,
        ),
    },

    {
        "name": "Random Forest",
        "features": "raw",
        "factory": lambda: RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            max_features=1.0,
            min_samples_leaf=2,
            criterion="gini",
            random_state=SEED,
            n_jobs=-1,
        ),
    },

    {
        "name": "Gradient Boosting",
        "features": "raw",
        "factory": lambda: GradientBoostingClassifier(
            n_estimators=100,
            learning_rate=0.1,
            max_depth=3,
            random_state=SEED,
        ),
    },

    {
        "name": "Extra Trees raw",
        "features": "raw",
        "factory": lambda: ExtraTreesClassifier(
            n_estimators=100,
            max_depth=None,
            max_features=1.0,
            min_samples_leaf=2,
            criterion="gini",
            random_state=SEED,
            n_jobs=-1,
        ),
    },

    {
        "name": "Extra Trees pair",
        "features": "pair",
        "factory": lambda: ExtraTreesClassifier(
            n_estimators=100,
            max_depth=None,
            max_features=1.0,
            min_samples_leaf=2,
            criterion="gini",
            random_state=SEED,
            n_jobs=-1,
        ),
    },

    {
        "name": "Logistic raw",
        "features": "raw",
        "factory": lambda: Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                C=0.003,
                solver="liblinear",
                max_iter=5000,
                random_state=SEED,
            )),
        ]),
    },

    {
        "name": "Logistic pair",
        "features": "pair",
        "factory": lambda: Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                C=0.003,
                solver="liblinear",
                max_iter=5000,
                random_state=SEED,
            )),
        ]),
    },

    {
        "name": "XGBoost raw",
        "features": "raw",
        "factory": lambda: XGBClassifier(
            n_estimators=400,
            max_depth=5,
            learning_rate=0.15,
            subsample=1.0,
            colsample_bytree=1.0,
            min_child_weight=1,
            random_state=SEED,
            n_jobs=-1,
            eval_metric="auc",
        ),
    },

    {
        "name": "XGBoost pair",
        "features": "pair",
        "factory": lambda: XGBClassifier(
            n_estimators=400,
            max_depth=5,
            learning_rate=0.15,
            subsample=1.0,
            colsample_bytree=1.0,
            min_child_weight=1,
            random_state=SEED,
            n_jobs=-1,
            eval_metric="auc",
        ),
    },

    {
        "name": "CatBoost raw",
        "features": "raw",
        "factory": lambda: CatBoostClassifier(
            iterations=2000,
            depth=6,
            learning_rate=0.05,
            l2_leaf_reg=3,
            random_strength=2,
            random_seed=SEED,
            verbose=False,
            allow_writing_files=False,
        ),
    },

    {
        "name": "CatBoost pair",
        "features": "pair",
        "factory": lambda: CatBoostClassifier(
            iterations=2000,
            depth=6,
            learning_rate=0.05,
            l2_leaf_reg=3,
            random_strength=2,
            random_seed=SEED,
            verbose=False,
            allow_writing_files=False,
        ),
    },

    {
        "name": "LightGBM raw",
        "features": "raw",
        "factory": lambda: LGBMClassifier(
            boosting_type="goss",
            colsample_bytree=0.7,
            learning_rate=0.05,
            max_depth=5,
            min_child_samples=20,
            n_estimators=500,
            num_leaves=31,
            reg_lambda=0.05,
            random_state=SEED,
            n_jobs=-1,
            verbosity=-1,
        ),
    },

    {
        "name": "LightGBM pair",
        "features": "pair",
        "factory": lambda: LGBMClassifier(
            boosting_type="goss",
            colsample_bytree=0.7,
            learning_rate=0.05,
            max_depth=5,
            min_child_samples=20,
            n_estimators=500,
            num_leaves=31,
            reg_lambda=0.05,
            random_state=SEED,
            n_jobs=-1,
            verbosity=-1,
        ),
    },
]


### 4. Вычисление Out-Of-Fold (OOF) предсказаний


In [4]:
oof_predictions = {}

cv_results = []

for model_number, spec in enumerate(model_specs, start=1):
    name = spec["name"]
    feature_name = spec["features"]

    X = feature_sets[feature_name]

    oof = np.zeros(len(y))

    fold_scores = []

    start = time.perf_counter()

    print(f"\n[{model_number}/{len(model_specs)}] {name}")

    for fold, (train_idx, val_idx) in enumerate(cv, start=1):
        model = spec["factory"]()

        model.fit(
            X[train_idx],
            y[train_idx],
        )

        prediction = model.predict_proba(
            X[val_idx]
        )[:, 1]

        oof[val_idx] = prediction

        auc = roc_auc_score(
            y[val_idx],
            prediction,
        )

        fold_scores.append(auc)

        print(f"fold {fold}: {auc:.6f}")

    elapsed = time.perf_counter() - start

    oof_predictions[name] = oof

    cv_results.append({
        "model": name,
        "mean_auc": np.mean(fold_scores),
        "std_auc": np.std(fold_scores),
        "fold_1": fold_scores[0],
        "fold_2": fold_scores[1],
        "fold_3": fold_scores[2],
        "fold_4": fold_scores[3],
        "fold_5": fold_scores[4],
        "time_sec": elapsed,
    })



[1/13] Decision Tree
fold 1: 0.769353
fold 2: 0.783856
fold 3: 0.794218
fold 4: 0.785116
fold 5: 0.786360

[2/13] Random Forest
fold 1: 0.892739
fold 2: 0.896432
fold 3: 0.879425
fold 4: 0.893085
fold 5: 0.904730

[3/13] Gradient Boosting
fold 1: 0.885062
fold 2: 0.884448
fold 3: 0.875803
fold 4: 0.894589
fold 5: 0.901817

[4/13] Extra Trees raw
fold 1: 0.894046
fold 2: 0.901754
fold 3: 0.893778
fold 4: 0.877929
fold 5: 0.913974

[5/13] Extra Trees pair
fold 1: 0.888070
fold 2: 0.902738
fold 3: 0.894794
fold 4: 0.875000
fold 5: 0.902353

[6/13] Logistic raw
fold 1: 0.896841
fold 2: 0.909596
fold 3: 0.907329
fold 4: 0.910226
fold 5: 0.918651

[7/13] Logistic pair
fold 1: 0.895739
fold 2: 0.907770
fold 3: 0.910147
fold 4: 0.905959
fold 5: 0.914730

[8/13] XGBoost raw
fold 1: 0.901140
fold 2: 0.905502
fold 3: 0.896857
fold 4: 0.900967
fold 5: 0.916950

[9/13] XGBoost pair
fold 1: 0.892967
fold 2: 0.904494
fold 3: 0.894637
fold 4: 0.896211
fold 5: 0.912950

[10/13] CatBoost raw
fold 1: 0.

### 5. Сводная таблица результатов всех моделей на 5-Fold CV


In [5]:
cv_results_df = (
    pd.DataFrame(cv_results)
    .sort_values(
        "mean_auc",
        ascending=False,
    )
    .reset_index(drop=True)
)

cv_results_df


,model,mean_auc,std_auc,fold_1,fold_2,fold_3,fold_4,fold_5,time_sec
0,LightGBM raw,0.913272,0.008291,0.902872,0.917816,0.912132,0.907045,0.926493,8.671915
1,Logistic raw,0.908529,0.006993,0.896841,0.909596,0.907329,0.910226,0.918651,0.207880
2,Logistic pair,0.906869,0.006294,0.895739,0.907770,0.910147,0.905959,0.914730,0.375888
3,LightGBM pair,0.905398,0.007104,0.895188,0.909990,0.904856,0.901140,0.915816,9.502783
4,CatBoost raw,0.904809,0.007565,0.896007,0.910242,0.897172,0.904825,0.915801,70.622927
5,XGBoost raw,0.904283,0.006899,0.901140,0.905502,0.896857,0.900967,0.916950,7.753396
6,CatBoost pair,0.901376,0.005177,0.894889,0.902494,0.899802,0.899187,0.910510,100.265877
7,XGBoost pair,0.900252,0.007486,0.892967,0.904494,0.894637,0.896211,0.912950,10.633699
8,Extra Trees raw,0.896296,0.011756,0.894046,0.901754,0.893778,0.877929,0.913974,2.183278
9,Random Forest,0.893282,0.008165,0.892739,0.896432,0.879425,0.893085,0.904730,58.778034


### 6. Функция оценки блендов (фолд-за-фолдом)


In [6]:
def evaluate_blend(prediction):
    fold_scores = []

    for fold in range(5):
        mask = fold_id == fold

        score = roc_auc_score(
            y[mask],
            prediction[mask],
        )

        fold_scores.append(score)

    return {
        "mean_auc": np.mean(fold_scores),
        "std_auc": np.std(fold_scores),
        "fold_scores": fold_scores,
    }


### 7. Ранжирование внутри каждого фолда (Rank Averaging)


In [7]:
def rank_oof(prediction):
    ranked = np.zeros_like(
        prediction,
        dtype=float,
    )

    for fold in range(5):
        mask = fold_id == fold

        ranked[mask] = (
            pd.Series(prediction[mask])
            .rank(
                method="average",
                pct=True,
            )
            .to_numpy()
        )

    return ranked


### 8. Отбор сильных моделей для блендинга


In [8]:
strong_models = (
    cv_results_df[
        cv_results_df["mean_auc"] >= 0.88
    ]["model"]
    .tolist()
)

strong_models


['LightGBM raw',
 'Logistic raw',
 'Logistic pair',
 'LightGBM pair',
 'CatBoost raw',
 'XGBoost raw',
 'CatBoost pair',
 'XGBoost pair',
 'Extra Trees raw',
 'Random Forest',
 'Extra Trees pair',
 'Gradient Boosting']

### 9. Перебор комбинаций моделей (Probability Averaging + Rank Averaging)


In [9]:
import itertools

blend_results = []

for size in range(
    2,
    len(strong_models) + 1,
):

    for combination in itertools.combinations(
        strong_models,
        size,
    ):

        # Probability averaging
        probability_prediction = np.mean(
            [
                oof_predictions[name]
                for name in combination
            ],
            axis=0,
        )

        probability_result = evaluate_blend(
            probability_prediction
        )

        blend_results.append({
            "method": "probability",
            "models": combination,
            "n_models": size,
            "mean_auc": probability_result["mean_auc"],
            "std_auc": probability_result["std_auc"],
        })


        # Rank averaging
        rank_prediction = np.mean(
            [
                rank_oof(
                    oof_predictions[name]
                )
                for name in combination
            ],
            axis=0,
        )

        rank_result = evaluate_blend(
            rank_prediction
        )

        blend_results.append({
            "method": "rank",
            "models": combination,
            "n_models": size,
            "mean_auc": rank_result["mean_auc"],
            "std_auc": rank_result["std_auc"],
        })


### 10. Настоящий рейтинг блендов (Mean CV ROC-AUC ± Std)


In [11]:
blend_cv_df = (
    pd.DataFrame(blend_results)
    .sort_values(
        "mean_auc",
        ascending=False,
    )
    .reset_index(drop=True)
)

blend_cv_df.head(20)


,method,models,n_models,mean_auc,std_auc
0,rank,"(LightGBM raw, Logistic pair)",2,0.916295,0.006599
1,probability,"(LightGBM raw, Logistic pair)",2,0.916046,0.006016
2,rank,"(LightGBM raw, Logistic pair, XGBoost raw)",3,0.915920,0.006757
3,probability,"(LightGBM raw, Logistic pair, Extra Trees raw)",3,0.915838,0.007022
4,probability,"(LightGBM raw, Logistic pair, Extra Trees pair)",3,0.915747,0.006440
5,rank,"(LightGBM raw, Logistic raw)",2,0.915646,0.007500
6,rank,"(LightGBM raw, Logistic raw, Logistic pair, XG...",4,0.915626,0.006687
7,probability,"(LightGBM raw, Logistic raw, Extra Trees raw)",3,0.915495,0.007401
8,probability,"(LightGBM raw, Logistic raw)",2,0.915492,0.006755
9,rank,"(LightGBM raw, Logistic pair, CatBoost raw)",3,0.915464,0.006740


In [12]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

seeds = [
    20260916,
    42,
    1337,
    2026,
    777,
]

repeat_results = []

for seed in seeds:

    cv_repeat = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=seed,
    )

    lgbm_scores = []
    logistic_scores = []
    rank_blend_scores = []
    probability_blend_scores = []

    for train_idx, val_idx in cv_repeat.split(X_raw, y):

        # LightGBM raw
        lgbm = LGBMClassifier(
            boosting_type="goss",
            colsample_bytree=0.7,
            learning_rate=0.05,
            max_depth=5,
            min_child_samples=20,
            n_estimators=500,
            num_leaves=31,
            reg_lambda=0.05,
            random_state=SEED,
            n_jobs=-1,
            verbosity=-1,
        )

        lgbm.fit(
            X_raw[train_idx],
            y[train_idx],
        )

        lgbm_pred = lgbm.predict_proba(
            X_raw[val_idx]
        )[:, 1]


        # Logistic pair
        logistic = Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                C=0.003,
                solver="liblinear",
                max_iter=5000,
                random_state=SEED,
            )),
        ])

        logistic.fit(
            X_pair[train_idx],
            y[train_idx],
        )

        logistic_pred = logistic.predict_proba(
            X_pair[val_idx]
        )[:, 1]


        # Probability blend
        probability_blend = (
            0.5 * lgbm_pred
            + 0.5 * logistic_pred
        )


        # Rank blend
        lgbm_rank = (
            pd.Series(lgbm_pred)
            .rank(pct=True)
            .to_numpy()
        )

        logistic_rank = (
            pd.Series(logistic_pred)
            .rank(pct=True)
            .to_numpy()
        )

        rank_blend = (
            0.5 * lgbm_rank
            + 0.5 * logistic_rank
        )


        y_fold = y[val_idx]

        lgbm_scores.append(
            roc_auc_score(y_fold, lgbm_pred)
        )

        logistic_scores.append(
            roc_auc_score(y_fold, logistic_pred)
        )

        probability_blend_scores.append(
            roc_auc_score(y_fold, probability_blend)
        )

        rank_blend_scores.append(
            roc_auc_score(y_fold, rank_blend)
        )


    repeat_results.append({
        "seed": seed,
        "lgbm_auc": np.mean(lgbm_scores),
        "logistic_auc": np.mean(logistic_scores),
        "probability_blend_auc": np.mean(
            probability_blend_scores
        ),
        "rank_blend_auc": np.mean(
            rank_blend_scores
        ),
    })


repeat_df = pd.DataFrame(repeat_results)

repeat_df

,seed,lgbm_auc,logistic_auc,probability_blend_auc,rank_blend_auc
0,20260916,0.913272,0.906869,0.916046,0.916295
1,42,0.912248,0.912380,0.917416,0.918884
2,1337,0.911410,0.910314,0.916323,0.917668
3,2026,0.910223,0.905867,0.912406,0.913717
4,777,0.907858,0.909467,0.914276,0.915155


In [14]:
summary = pd.DataFrame({
    "model": [
        "LightGBM raw",
        "Logistic pair",
        "Probability blend",
        "Rank blend",
    ],
    "mean_auc": [
        repeat_df["lgbm_auc"].mean(),
        repeat_df["logistic_auc"].mean(),
        repeat_df["probability_blend_auc"].mean(),
        repeat_df["rank_blend_auc"].mean(),
    ],
    "std_auc": [
        repeat_df["lgbm_auc"].std(),
        repeat_df["logistic_auc"].std(),
        repeat_df["probability_blend_auc"].std(),
        repeat_df["rank_blend_auc"].std(),
    ],
})

summary.sort_values(
    "mean_auc",
    ascending=False,
)

,model,mean_auc,std_auc
3,Rank blend,0.916344,0.002032
2,Probability blend,0.915294,0.001969
0,LightGBM raw,0.911002,0.002083
1,Logistic pair,0.908980,0.002633


# Final submission

По repeated cross-validation лучший стабильный результат дал rank blend:

- LightGBM на исходных признаках
- Logistic Regression на исходных признаках + `|A-B|`

Финальный blend: 50/50 по рангам.

In [15]:
# Final LightGBM

final_lgbm = LGBMClassifier(
    boosting_type="goss",
    colsample_bytree=0.7,
    learning_rate=0.05,
    max_depth=5,
    min_child_samples=20,
    n_estimators=500,
    num_leaves=31,
    reg_lambda=0.05,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
)

final_lgbm.fit(
    X_raw,
    y,
)

lgbm_test_probability = final_lgbm.predict_proba(
    X_test_raw
)[:, 1]

In [16]:
final_logistic = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        C=0.003,
        solver="liblinear",
        max_iter=5000,
        random_state=SEED,
    )),
])

final_logistic.fit(
    X_pair,
    y,
)

logistic_test_probability = final_logistic.predict_proba(
    X_test_pair
)[:, 1]

In [17]:
lgbm_test_rank = (
    pd.Series(lgbm_test_probability)
    .rank(method="average", pct=True)
    .to_numpy()
)

logistic_test_rank = (
    pd.Series(logistic_test_probability)
    .rank(method="average", pct=True)
    .to_numpy()
)

final_test_probability = (
    0.5 * lgbm_test_rank
    + 0.5 * logistic_test_rank
)

In [18]:
submission = sample_submission.copy()

submission["target"] = final_test_probability

submission.to_csv(
    "../submission_cv_rank_lgbm_logistic.csv",
    index=False,
)

print(submission.head())
print("Rows:", len(submission))
print("NaN:", submission["target"].isna().sum())
print("Min:", submission["target"].min())
print("Max:", submission["target"].max())

                     row_id    target
0  row_46ad1afad8562239f79c  0.840950
1  row_b42ade9cd57b8d919acb  0.782258
2  row_c951cd5e197b6b4e8b83  0.390233
3  row_a54b3423760d599416ce  0.063620
4  row_a87c87cc47690e2a303f  0.872312
Rows: 1116
NaN: 0
Min: 0.004032258064516129
Max: 0.9896953405017921
